# 🤖 Mini-Projet NLP — Partie 2 : Système RAG
**ENSA Al Hoceima — Ingénierie des Données 2ème Année**

---

## 📌 Pipeline RAG Implémenté

| Étape | Description | Outil |
|-------|-------------|-------|
| 1 | Encodage des documents | `sentence-transformers` |
| 2 | Indexation vectorielle | `FAISS IndexFlatIP` |
| 3 | Encodage de la requête | `sentence-transformers` |
| 4 | Recherche top-k documents | `FAISS .search()` |
| 5 | Construction du prompt augmenté | Python |
| 6 | Génération de la réponse | `flan-t5` |

---

In [ ]:
# =============================================
# CELLULE 1 — Installation des dépendances
# =============================================
!pip install sentence-transformers faiss-cpu transformers accelerate -q
print('✅ Installation terminée')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 41.9 MB/s eta 0:00:00
✅ Installation terminée


In [ ]:
# =============================================
# CELLULE 2 — Imports
# =============================================
import os
import re
import time
import textwrap
import numpy as np
from typing import List, Tuple, Dict

import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print('✅ Toutes les bibliothèques importées avec succès')

✅ Toutes les bibliothèques importées avec succès


## 📚 Corpus de documents
Nous utilisons un corpus sur les thèmes NLP/Transformers. En pratique, ce corpus peut être remplacé par des PDFs, articles, ou tout dataset.

In [ ]:
# =============================================
# CELLULE 3 — Corpus de documents
# =============================================
CORPUS = [
    {
        'id': 0,
        'title': 'Introduction aux Transformers',
        'text': (
            'Les Transformers sont une architecture de réseau de neurones introduite en 2017 '
            "dans l'article 'Attention is All You Need' par Vaswani et al. "
            "Ils reposent sur un mécanisme d'attention qui permet au modèle de pondérer "
            "l'importance de chaque mot dans une séquence par rapport aux autres. "
            'Contrairement aux RNN et LSTM, les Transformers traitent tous les tokens en parallèle, '
            'ce qui les rend beaucoup plus efficaces à entraîner sur des GPU modernes. '
            "Les composants principaux sont : l'encodeur, le décodeur, et le mécanisme d'auto-attention multi-têtes."
        ),
    },
    {
        'id': 1,
        'title': 'BERT et les modèles encodeurs',
        'text': (
            'BERT (Bidirectional Encoder Representations from Transformers) est un modèle '
            'pré-entraîné par Google en 2018. Il utilise uniquement la partie encodeur des Transformers. '
            'BERT est pré-entraîné sur deux tâches : le Masked Language Modeling (MLM) où 15% des tokens '
            'sont masqués et le modèle doit les prédire, et la Next Sentence Prediction (NSP). '
            'BERT excelle dans les tâches de compréhension du langage : classification de texte, '
            'reconnaissance d\'entités nommées (NER), et question answering extractif. '
            'Des variantes incluent RoBERTa, DistilBERT, CamemBERT (pour le français).'
        ),
    },
    {
        'id': 2,
        'title': 'GPT et les modèles génératifs',
        'text': (
            'GPT (Generative Pre-trained Transformer) est une série de modèles développés par OpenAI '
            'utilisant uniquement la partie décodeur des Transformers. GPT-1 (2018), GPT-2 (2019), '
            'GPT-3 (2020) avec 175 milliards de paramètres, et GPT-4 (2023) ont marqué des étapes importantes. '
            'Ces modèles sont entraînés par prédiction du token suivant (autoregressive). '
            'Ils excellent dans la génération de texte, la traduction, le résumé et le code. '
            'ChatGPT est une version fine-tunée de GPT avec RLHF (Reinforcement Learning from Human Feedback).'
        ),
    },
    {
        'id': 3,
        'title': 'RAG — Retrieval-Augmented Generation',
        'text': (
            'Le RAG (Retrieval-Augmented Generation) est une architecture qui combine la recherche '
            "d'information avec la génération de texte par LLM. Proposé par Lewis et al. (2020) chez Facebook AI, "
            "le RAG résout les limitations des LLMs : hallucinations, connaissances figées, et manque de sources. "
            "Le pipeline RAG se déroule en plusieurs étapes : indexation des documents sous forme d'embeddings, "
            "encodage de la requête utilisateur, recherche des k documents les plus similaires via similarité cosinus, "
            "construction d'un prompt augmenté combinant la requête et le contexte récupéré, "
            'puis génération de la réponse finale par le LLM. '
            'Les avantages du RAG incluent : réponses plus précises, sources vérifiables, et mise à jour facile du corpus.'
        ),
    },
    {
        'id': 4,
        'title': 'FAISS — Recherche vectorielle',
        'text': (
            'FAISS (Facebook AI Similarity Search) est une bibliothèque open-source développée par Meta '
            'pour la recherche efficace de similarité dans des espaces vectoriels de grande dimension. '
            'FAISS supporte plusieurs types d\'index : IndexFlatL2 (recherche exacte par distance L2), '
            'IndexFlatIP (produit scalaire / cosinus), IndexIVFFlat (recherche approximative plus rapide). '
            'Pour normaliser les vecteurs et utiliser la similarité cosinus avec IndexFlatIP, '
            'il faut d\'abord normaliser les embeddings avec faiss.normalize_L2(). '
            'FAISS peut gérer des milliards de vecteurs et s\'exécute sur CPU et GPU.'
        ),
    },
    {
        'id': 5,
        'title': 'Sentence Transformers et embeddings sémantiques',
        'text': (
            'Les Sentence Transformers (SBERT) sont des modèles spécialement fine-tunés pour produire '
            'des embeddings de phrases sémantiquement riches. Contrairement à BERT qui encode token par token, '
            "SBERT produit un vecteur dense représentant le sens global d'une phrase. "
            "Le modèle 'all-MiniLM-L6-v2' produit des vecteurs de dimension 384, léger et performant. "
            "Le modèle 'paraphrase-multilingual-MiniLM-L12-v2' supporte plus de 50 langues dont le français et l'arabe. "
            'Ces embeddings sont utilisés pour la recherche sémantique, la détection de paraphrases, '
            'le clustering de documents, et les systèmes RAG.'
        ),
    },
    {
        'id': 6,
        'title': 'Fine-tuning vs Prompt Engineering vs RAG',
        'text': (
            'Il existe trois approches principales pour adapter les LLMs à des domaines spécifiques. '
            'Le Fine-tuning consiste à ré-entraîner le modèle sur des données spécifiques : très performant '
            'mais coûteux en calcul, données et temps. Le Prompt Engineering consiste à guider le modèle '
            'par des instructions précises dans le prompt : rapide et sans coût d\'entraînement, '
            'mais limité par la fenêtre de contexte et les connaissances figées du modèle. '
            'Le RAG combine les deux avantages : pas de ré-entraînement, accès à des connaissances actualisées, '
            'réponses traçables avec sources. RAG est particulièrement adapté pour les bases de connaissances '
            'évolutives, les chatbots d\'entreprise, et les applications nécessitant de la précision factuelle.'
        ),
    },
    {
        'id': 7,
        'title': 'T5 et les modèles séquence-à-séquence',
        'text': (
            'T5 (Text-to-Text Transfer Transformer) est un modèle développé par Google Research en 2019. '
            'Son innovation principale est de reformuler toutes les tâches NLP comme des problèmes de '
            'transformation texte-vers-texte. Par exemple, la traduction devient \'translate English to French: ...\' '
            'et la classification devient une génération de label. T5 utilise l\'architecture encodeur-décodeur complète. '
            "La version 'flan-t5' est une version améliorée avec instruction tuning, disponible en plusieurs tailles."
        ),
    },
]

print(f'✅ Corpus chargé : {len(CORPUS)} documents')
for doc in CORPUS:
    print(f"   [{doc['id']}] {doc['title']}")

✅ Corpus chargé : 8 documents
   [0] Introduction aux Transformers
   [1] BERT et les modèles encodeurs
   [2] GPT et les modèles génératifs
   [3] RAG — Retrieval-Augmented Generation
   [4] FAISS — Recherche vectorielle
   [5] Sentence Transformers et embeddings sémantiques
   [6] Fine-tuning vs Prompt Engineering vs RAG
   [7] T5 et les modèles séquence-à-séquence


## 🏗️ Architecture RAG — Classe principale

In [ ]:
# =============================================
# CELLULE 4 — Classe RAGSystem complète
# =============================================

class RAGSystem:
    """
    Système RAG complet :
      - Sentence Transformers pour les embeddings
      - FAISS pour la recherche vectorielle
      - GPT2 pour la génération
    """

    def __init__(self, embedding_model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
                 generative_model_name='gpt2', top_k=3):
        self.top_k = top_k
        self.corpus = []
        self.index = None
        self.embeddings = None

        print('─' * 55)
        print('  Initialisation du Système RAG')
        print('─' * 55)

        # Modèle d'embeddings
        print(f'\n📦 Modèle d\'embeddings : {embedding_model_name}')
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.embedding_dim = self.embedding_model.get_embedding_dimension()
        print(f'   ✅ Dimension : {self.embedding_dim}')

        # Modèle génératif
        print(f'\n🤖 Modèle génératif : {generative_model_name}')
        self.generator = pipeline('text-generation', model=generative_model_name,
                                   max_new_tokens=200, truncation=True)
        print('   ✅ Prêt')
        print('─' * 55)

    # --- Étapes 1 & 2 : Encodage + Indexation FAISS ---
    def build_index(self, corpus):
        self.corpus = corpus
        texts = [doc['text'] for doc in corpus]

        print('\n📄 ÉTAPE 1 — Encodage des documents...')
        self.embeddings = self.embedding_model.encode(
            texts, show_progress_bar=True, normalize_embeddings=True
        )
        print(f'   ✅ Shape : {self.embeddings.shape}')

        print('\n🗃️  ÉTAPE 2 — Construction index FAISS...')
        self.index = faiss.IndexFlatIP(self.embedding_dim)
        self.index.add(self.embeddings.astype(np.float32))
        print(f'   ✅ {self.index.ntotal} vecteurs indexés')

    # --- Étapes 3 & 4 : Récupération ---
    def retrieve(self, query):
        print(f'\n🔍 ÉTAPE 3 — Encodage requête...')
        q_emb = self.embedding_model.encode([query], normalize_embeddings=True).astype(np.float32)

        print(f'🎯 ÉTAPE 4 — Recherche top-{self.top_k}...')
        scores, indices = self.index.search(q_emb, self.top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            doc = self.corpus[idx]
            results.append((doc, float(score)))
            print(f'   [{idx}] {score:.4f} | {doc["title"]}')
        return results

    # --- Étape 5 : Prompt augmenté ---
    def build_augmented_prompt(self, query, retrieved_docs):
        context_parts = []
        for i, (doc, score) in enumerate(retrieved_docs, 1):
            context_parts.append(f"Source {i} — {doc['title']}:\n{doc['text']}")
        context = '\n\n'.join(context_parts)
        prompt = (
            f"Contexte:\n{context}\n\n"
            f"Question: {query}\n\n"
            f"Réponse:"
        )
        print(f'\n📝 ÉTAPE 5 — Prompt augmenté ({len(prompt)} caractères)')
        return prompt

    # --- Étape 6 : Génération ---
    def generate_response(self, prompt):
        print('\n🧠 ÉTAPE 6 — Génération de la réponse...')
        output = self.generator(prompt, max_new_tokens=200, do_sample=False, truncation=True)
        full_text = output[0]['generated_text']
        response = full_text.split('Réponse:')[-1].strip()
        return response if response else full_text[:300]

    # --- Pipeline complet ---
    def query(self, question, verbose=True):
        if verbose:
            print('\n' + '═' * 55)
            print(f'  REQUÊTE : {question}')
            print('═' * 55)
        retrieved = self.retrieve(question)
        prompt = self.build_augmented_prompt(question, retrieved)
        response = self.generate_response(prompt)
        return {
            'question': question,
            'response': response,
            'sources': [{'title': d['title'], 'score': round(s, 4), 'id': d['id']} for d, s in retrieved],
            'prompt': prompt,
        }

print('✅ Classe RAGSystem définie')

✅ Classe RAGSystem définie


## 🚀 Initialisation et indexation

In [ ]:
# =============================================
# CELLULE 5 — Initialisation + Build index
# =============================================
rag = RAGSystem(
    embedding_model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    generative_model_name='google/flan-t5-small',
    top_k=3,
)
rag.build_index(CORPUS)

───────────────────────────────────────────────────────
  Initialisation du Système RAG
───────────────────────────────────────────────────────

📦 Modèle d'embeddings : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ✅ Dimension : 384

🤖 Modèle génératif : google/flan-t5-small


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

   ✅ Prêt
───────────────────────────────────────────────────────

📄 ÉTAPE 1 — Encodage des documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   ✅ Shape : (8, 384)

🗃️  ÉTAPE 2 — Construction index FAISS...
   ✅ 8 vecteurs indexés


## 🧪 Démonstration — Comparaison avec / sans RAG

In [ ]:
# =============================================
# CELLULE 6 — Comparaison RAG vs sans RAG
# =============================================

def generate_without_rag(generator, question):
    """Baseline : génération directe sans contexte récupéré."""
    prompt = f'Question: {question}\nRéponse:'
    output = generator(prompt, max_new_tokens=150, do_sample=False)
    return output[0]['generated_text'].strip()


TEST_QUESTIONS = [
    'Qu\'est-ce que FAISS et comment fonctionne-t-il ?',
    'Quelle est la différence entre BERT et GPT ?',
    'Quels sont les avantages du RAG par rapport au fine-tuning ?',
    'Comment fonctionne Sentence Transformers ?',
]

results_log = []

for i, question in enumerate(TEST_QUESTIONS, 1):
    print(f'\n{"─" * 55}')
    print(f'Question {i}: {question}')
    print('─' * 55)

    # Sans RAG
    response_no_rag = generate_without_rag(rag.generator, question)
    print(f'\n❌ SANS RAG :\n{response_no_rag[:200]}')

    # Avec RAG
    result = rag.query(question, verbose=False)
    print(f'\n✅ AVEC RAG :\n{result["response"][:200]}')
    print(f'\n📚 Sources : {[s["title"] for s in result["sources"]]}')

    results_log.append({
        'question': question,
        'without_rag': response_no_rag,
        'with_rag': result['response'],
        'sources': result['sources'],
    })

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



───────────────────────────────────────────────────────
Question 1: Qu'est-ce que FAISS et comment fonctionne-t-il ?
───────────────────────────────────────────────────────

❌ SANS RAG :
Question: Qu'est-ce que FAISS et comment fonctionne-t-il ?
Réponse:

🔍 ÉTAPE 3 — Encodage requête...
🎯 ÉTAPE 4 — Recherche top-3...
   [0] 0.3660 | Introduction aux Transformers
   [7] 0.3123 | T5 et les modèles séquence-à-séquence
   [1] 0.2990 | BERT et les modèles encodeurs

📝 ÉTAPE 5 — Prompt augmenté (1837 caractères)

🧠 ÉTAPE 6 — Génération de la réponse...


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ AVEC RAG :
Contexte:
Source 1 — Introduction aux Transformers:
Les Transformers sont une architecture de réseau de neurones introduite en 2017 dans l'article 'Attention is All You Need' par Vaswani et al. Ils re

📚 Sources : ['Introduction aux Transformers', 'T5 et les modèles séquence-à-séquence', 'BERT et les modèles encodeurs']

───────────────────────────────────────────────────────
Question 2: Quelle est la différence entre BERT et GPT ?
───────────────────────────────────────────────────────

❌ SANS RAG :
Question: Quelle est la différence entre BERT et GPT ?
Réponse:

🔍 ÉTAPE 3 — Encodage requête...
🎯 ÉTAPE 4 — Recherche top-3...
   [2] 0.5913 | GPT et les modèles génératifs
   [1] 0.4184 | BERT et les modèles encodeurs
   [7] 0.3784 | T5 et les modèles séquence-à-séquence

📝 ÉTAPE 5 — Prompt augmenté (1809 caractères)

🧠 ÉTAPE 6 — Génération de la réponse...


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ AVEC RAG :
Contexte:
Source 1 — GPT et les modèles génératifs:
GPT (Generative Pre-trained Transformer) est une série de modèles développés par OpenAI utilisant uniquement la partie décodeur des Transformers. GP

📚 Sources : ['GPT et les modèles génératifs', 'BERT et les modèles encodeurs', 'T5 et les modèles séquence-à-séquence']

───────────────────────────────────────────────────────
Question 3: Quels sont les avantages du RAG par rapport au fine-tuning ?
───────────────────────────────────────────────────────

❌ SANS RAG :
Question: Quels sont les avantages du RAG par rapport au fine-tuning ?
Réponse:ces ; d) the RAG's reassurances ; d) the RAG's reassurances ; d) the RAG's reassurances ; d) the RAG's reassurances ; d) 

🔍 ÉTAPE 3 — Encodage requête...
🎯 ÉTAPE 4 — Recherche top-3...
   [6] 0.4947 | Fine-tuning vs Prompt Engineering vs RAG
   [3] 0.4550 | RAG — Retrieval-Augmented Generation
   [0] 0.2406 | Introduction aux Transformers

📝 ÉTAPE 5 — Prompt augmenté (2250 caractèr

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ AVEC RAG :
Contexte:
Source 1 — Fine-tuning vs Prompt Engineering vs RAG:
Il existe trois approches principales pour adapter les LLMs à des domaines spécifiques. Le Fine-tuning consiste à ré-entraîner le modèle 

📚 Sources : ['Fine-tuning vs Prompt Engineering vs RAG', 'RAG — Retrieval-Augmented Generation', 'Introduction aux Transformers']

───────────────────────────────────────────────────────
Question 4: Comment fonctionne Sentence Transformers ?
───────────────────────────────────────────────────────

❌ SANS RAG :
Question: Comment fonctionne Sentence Transformers ?
Réponse:

🔍 ÉTAPE 3 — Encodage requête...
🎯 ÉTAPE 4 — Recherche top-3...
   [0] 0.6597 | Introduction aux Transformers
   [5] 0.6415 | Sentence Transformers et embeddings sémantiques
   [1] 0.5622 | BERT et les modèles encodeurs

📝 ÉTAPE 5 — Prompt augmenté (1937 caractères)

🧠 ÉTAPE 6 — Génération de la réponse...


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ AVEC RAG :
Contexte:
Source 1 — Introduction aux Transformers:
Les Transformers sont une architecture de réseau de neurones introduite en 2017 dans l'article 'Attention is All You Need' par Vaswani et al. Ils re

📚 Sources : ['Introduction aux Transformers', 'Sentence Transformers et embeddings sémantiques', 'BERT et les modèles encodeurs']


## 💬 Test d'une requête personnalisée

In [ ]:
# =============================================
# CELLULE 7 — Requête personnalisée
# =============================================

ma_question = 'Comment les Transformers gèrent-ils les séquences longues ?'

result = rag.query(ma_question)

print('\n' + '═' * 55)
print('RÉPONSE FINALE :')
print('═' * 55)
print(result['response'])
print('\n📚 Sources utilisées :')
for src in result['sources']:
    print(f"  [{src['id']}] {src['title']} — score: {src['score']}")


═══════════════════════════════════════════════════════
  REQUÊTE : Comment les Transformers gèrent-ils les séquences longues ?
═══════════════════════════════════════════════════════

🔍 ÉTAPE 3 — Encodage requête...
🎯 ÉTAPE 4 — Recherche top-3...
   [0] 0.6874 | Introduction aux Transformers
   [5] 0.5531 | Sentence Transformers et embeddings sémantiques
   [1] 0.4900 | BERT et les modèles encodeurs

📝 ÉTAPE 5 — Prompt augmenté (1954 caractères)

🧠 ÉTAPE 6 — Génération de la réponse...


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



═══════════════════════════════════════════════════════
RÉPONSE FINALE :
═══════════════════════════════════════════════════════
Contexte:
Source 1 — Introduction aux Transformers:
Les Transformers sont une architecture de réseau de neurones introduite en 2017 dans l'article 'Attention is All You Need' par Vaswani et al. Ils reposent sur un mécanisme d'attention qui permet au modèle de pondérer l'importance de chaque mot dans

📚 Sources utilisées :
  [0] Introduction aux Transformers — score: 0.6874
  [5] Sentence Transformers et embeddings sémantiques — score: 0.5531
  [1] BERT et les modèles encodeurs — score: 0.49


## 💾 Sauvegarde de l'index FAISS

In [ ]:
# =============================================
# CELLULE 8 — Sauvegarde / Chargement index
# =============================================

# Sauvegarder
faiss.write_index(rag.index, './rag_index.faiss')
print('✅ Index FAISS sauvegardé : ./rag_index.faiss')

# Charger
loaded_index = faiss.read_index('./rag_index.faiss')
print(f'✅ Index chargé : {loaded_index.ntotal} vecteurs')

✅ Index FAISS sauvegardé : ./rag_index.faiss
✅ Index chargé : 8 vecteurs


## ✅ Résumé du système RAG implémenté

| Composant | Choix | Justification |
|-----------|-------|---------------|
| **Embeddings** | `paraphrase-multilingual-MiniLM-L12-v2` | Multilingue (FR/AR/EN), léger, efficace |
| **Index vectoriel** | `FAISS IndexFlatIP` | Similarité cosinus exacte, rapide |
| **Top-k** | 3 documents | Équilibre contexte / longueur du prompt |
| **Modèle génératif** | `google/flan-t5-small` | Open-source, instruction-tuned, CPU-friendly |
| **Corpus** | 8 documents NLP | Thématique cohérente avec le projet |

### 🔑 Points clés
- Les embeddings sont **normalisés** → similarité cosinus via produit scalaire
- Le prompt augmenté intègre les sources avec leur score de pertinence
- Le système est modulaire : corpus, modèle d'embeddings, et LLM sont facilement remplaçables